In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"data\processed\utci_analysis.csv"
)

In [2]:
df["time"] = pd.to_datetime(df["time"])

# ERA5 time is UTC → converting to Ahmedabad/India time
df["local_time"] = (
    df["time"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Asia/Kolkata")
)

df["local_time"] = df["local_time"].dt.tz_localize(None)

print(df[["time", "local_time"]].head(10))

                 time          local_time
0 2021-04-01 00:00:00 2021-04-01 05:30:00
1 2021-04-01 01:00:00 2021-04-01 06:30:00
2 2021-04-01 02:00:00 2021-04-01 07:30:00
3 2021-04-01 03:00:00 2021-04-01 08:30:00
4 2021-04-01 04:00:00 2021-04-01 09:30:00
5 2021-04-01 05:00:00 2021-04-01 10:30:00
6 2021-04-01 06:00:00 2021-04-01 11:30:00
7 2021-04-01 07:00:00 2021-04-01 12:30:00
8 2021-04-01 08:00:00 2021-04-01 13:30:00
9 2021-04-01 09:00:00 2021-04-01 14:30:00


In [3]:
df["hour"] = df["local_time"].dt.hour

df["period"] = np.where(
    (df["hour"] >= 6) & (df["hour"] < 18),
    "day",
    "night"
)

print(df[["local_time", "hour", "period"]].head(15))

print("\nPeriod counts:")
print(df["period"].value_counts())

            local_time  hour period
0  2021-04-01 05:30:00     5  night
1  2021-04-01 06:30:00     6    day
2  2021-04-01 07:30:00     7    day
3  2021-04-01 08:30:00     8    day
4  2021-04-01 09:30:00     9    day
5  2021-04-01 10:30:00    10    day
6  2021-04-01 11:30:00    11    day
7  2021-04-01 12:30:00    12    day
8  2021-04-01 13:30:00    13    day
9  2021-04-01 14:30:00    14    day
10 2021-04-01 15:30:00    15    day
11 2021-04-01 16:30:00    16    day
12 2021-04-01 17:30:00    17    day
13 2021-04-01 18:30:00    18  night
14 2021-04-01 19:30:00    19  night

Period counts:
period
night    58968
day      58968
Name: count, dtype: int64


In [4]:
night_df = df[df["period"] == "night"].copy()

print("Nighttime rows:", len(night_df))

print("\nNighttime UTCI:")
print(night_df["utci"].describe())

print("\nMissing nighttime UTCI:")
print(night_df["utci"].isna().sum())

Nighttime rows: 58968

Nighttime UTCI:
count    58968.000000
mean        29.538210
std          5.273055
min         13.948167
25%         26.329236
50%         29.347497
75%         31.780450
max         48.904745
Name: utci, dtype: float64

Missing nighttime UTCI:
0


In [5]:
night_df["date"] = night_df["local_time"].dt.date

night_daily = (
    night_df
    .groupby("date")["utci"]
    .mean()
    .reset_index()
)

night_daily["date"] = pd.to_datetime(night_daily["date"])

print(night_daily.head())
print(night_daily.shape)
print(night_daily["utci"].describe())

        date       utci
0 2021-04-01  25.449822
1 2021-04-02  23.045529
2 2021-04-03  24.180063
3 2021-04-04  25.795648
4 2021-04-05  26.839625
(552, 2)
count    552.000000
mean      29.521642
std        2.761781
min       22.042710
25%       27.344519
50%       30.221228
75%       31.724934
max       34.387933
Name: utci, dtype: float64


In [6]:
night_daily["year"] = night_daily["date"].dt.year
night_daily["month"] = night_daily["date"].dt.month

print(
    night_daily.groupby("year")["date"]
    .agg(["min", "max", "count"])
)

            min        max  count
year                             
2021 2021-04-01 2021-07-01     92
2022 2022-04-01 2022-07-01     92
2023 2023-04-01 2023-07-01     92
2024 2024-04-01 2024-07-01     92
2025 2025-04-01 2025-07-01     92
2026 2026-04-01 2026-07-01     92


In [7]:
print(
    night_daily.groupby(["year", "month"])
    .size()
    .unstack(fill_value=0)
)

month   4   5   6  7
year                
2021   30  31  30  1
2022   30  31  30  1
2023   30  31  30  1
2024   30  31  30  1
2025   30  31  30  1
2026   30  31  30  1


In [8]:
# Keep only April, May, June
night_daily = night_daily[
    night_daily["month"].isin([4, 5, 6])
].copy()

print(
    night_daily.groupby(["year", "month"])
    .size()
    .unstack(fill_value=0)
)

month   4   5   6
year             
2021   30  31  30
2022   30  31  30
2023   30  31  30
2024   30  31  30
2025   30  31  30
2026   30  31  30


In [9]:

years = sorted(night_daily["year"].unique())

baseline_list = []

for target_year in years:

    historical_data = night_daily[
        night_daily["year"] < target_year
    ]

    # Cannot calculate a historical baseline for the first year without earlier data
    if historical_data.empty:
        continue

    yearly_baseline = (
        historical_data
        .groupby("month")["utci"]
        .median()
        .reset_index(name="nighttime_baseline_utci")
    )

    yearly_baseline["target_year"] = target_year

    baseline_list.append(yearly_baseline)

rolling_baseline = pd.concat(
    baseline_list,
    ignore_index=True
)

print(rolling_baseline)

    month  nighttime_baseline_utci  target_year
0       4                26.497559         2022
1       5                30.449985         2022
2       6                30.934785         2022
3       4                26.591949         2023
4       5                30.658364         2023
5       6                31.409415         2023
6       4                26.550449         2024
7       5                30.541297         2024
8       6                31.383886         2024
9       4                26.622943         2025
10      5                30.698399         2025
11      6                31.651980         2025
12      4                26.591949         2026
13      5                30.799968         2026
14      6                31.558613         2026


In [10]:

night_excess_all = []

for target_year in rolling_baseline["target_year"].unique():

    # Nighttime data for the target year
    yearly_night = night_daily[
        night_daily["year"] == target_year
    ].copy()

    # Baseline for that target year
    yearly_baseline = rolling_baseline[
        rolling_baseline["target_year"] == target_year
    ][
        ["month", "nighttime_baseline_utci"]
    ]

    # Merge baseline onto nighttime data
    yearly_night = yearly_night.merge(
        yearly_baseline,
        on="month",
        how="left"
    )

    # Calculate excess above historical baseline
    yearly_night["nighttime_excess"] = (
        yearly_night["utci"]
        - yearly_night["nighttime_baseline_utci"]
    )

    night_excess_all.append(yearly_night)


# Combine all years
night_excess_all = pd.concat(
    night_excess_all,
    ignore_index=True
)

# Sort chronologically
night_excess_all = (
    night_excess_all
    .sort_values("date")
    .reset_index(drop=True)
)

print(
    night_excess_all[
        [
            "date",
            "year",
            "month",
            "utci",
            "nighttime_baseline_utci",
            "nighttime_excess"
        ]
    ].head(20)
)

         date  year  month       utci  nighttime_baseline_utci  \
0  2022-04-01  2022      4  28.747911                26.497559   
1  2022-04-02  2022      4  26.416519                26.497559   
2  2022-04-03  2022      4  26.009136                26.497559   
3  2022-04-04  2022      4  24.580740                26.497559   
4  2022-04-05  2022      4  26.617223                26.497559   
5  2022-04-06  2022      4  25.405756                26.497559   
6  2022-04-07  2022      4  24.503725                26.497559   
7  2022-04-08  2022      4  25.833770                26.497559   
8  2022-04-09  2022      4  26.013463                26.497559   
9  2022-04-10  2022      4  25.234353                26.497559   
10 2022-04-11  2022      4  26.307991                26.497559   
11 2022-04-12  2022      4  24.791119                26.497559   
12 2022-04-13  2022      4  24.510248                26.497559   
13 2022-04-14  2022      4  25.121394                26.497559   
14 2022-04

In [11]:

nctl_results = []

for year in sorted(night_excess_all["year"].unique()):

    yearly_data = (
        night_excess_all[
            night_excess_all["year"] == year
        ]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    nctl = []

    for i in range(len(yearly_data)):

        if i == 0:
            previous_nctl = 0.0
        else:
            previous_nctl = nctl[i - 1]

        current_excess = yearly_data.loc[
            i, "nighttime_excess"
        ]

        current_nctl = max(
            0.0,
            previous_nctl + current_excess
        )

        nctl.append(current_nctl)

    yearly_data["NCTL"] = nctl

    nctl_results.append(yearly_data)


# Combine all years
nctl_all = pd.concat(
    nctl_results,
    ignore_index=True
)

nctl_all = (
    nctl_all
    .sort_values(["year", "date"])
    .reset_index(drop=True)
)

print(
    nctl_all[
        [
            "date",
            "year",
            "utci",
            "nighttime_baseline_utci",
            "nighttime_excess",
            "NCTL"
        ]
    ].head(30)
)

         date  year       utci  nighttime_baseline_utci  nighttime_excess  \
0  2022-04-01  2022  28.747911                26.497559          2.250352   
1  2022-04-02  2022  26.416519                26.497559         -0.081039   
2  2022-04-03  2022  26.009136                26.497559         -0.488422   
3  2022-04-04  2022  24.580740                26.497559         -1.916819   
4  2022-04-05  2022  26.617223                26.497559          0.119665   
5  2022-04-06  2022  25.405756                26.497559         -1.091803   
6  2022-04-07  2022  24.503725                26.497559         -1.993833   
7  2022-04-08  2022  25.833770                26.497559         -0.663788   
8  2022-04-09  2022  26.013463                26.497559         -0.484095   
9  2022-04-10  2022  25.234353                26.497559         -1.263205   
10 2022-04-11  2022  26.307991                26.497559         -0.189568   
11 2022-04-12  2022  24.791119                26.497559         -1.706440   

In [12]:
print(
    nctl_all.groupby("year")["NCTL"].agg(
        ["min", "max", "mean"]
    )
)

      min        max       mean
year                           
2022  0.0  53.417096  20.954023
2023  0.0  12.130954   3.007065
2024  0.0  82.237251  29.396970
2025  0.0  38.532263  16.720089
2026  0.0  34.934227  13.003752
